# 05 — Review-budget routing (RQ4)
Ranks items by each score and sends the top-k% to human review; reports recall of real misregistrations at each budget, and misregistrations caught per review. Compares GUARD-derived item scores against individual baselines.

In [1]:
# Notebook: 05_routing_policy
# Shared plotting style: grayscale seaborn, dpi 600, PNG + PDF, no captions.
import os, numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
plt.rcParams["axes.edgecolor"] = "0.2"; plt.rcParams["axes.linewidth"] = 0.8
plt.rcParams["font.family"] = "DejaVu Sans"
GREYS = ["#111111", "#555555", "#888888", "#bbbbbb", "#dddddd"]
FIG = os.path.join("..", "results", "figures"); TAB = os.path.join("..", "results", "tables")
os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)
def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG, f"{name}.{ext}"), dpi=600, bbox_inches="tight")
    plt.close(fig)

import sys; sys.path.append(os.path.join("..", "src"))
import numpy as np
from guard_core import review_budget_curve
DATA_DIR = os.path.join("..", "data")
meta = pd.read_parquet(os.path.join(DATA_DIR, "item_meta.parquet"))
pi = np.load(os.path.join(DATA_DIR, "pi_memmap.npy"), mmap_mode="r")
N, K = pi.shape
valid = meta["clean_id"].values >= 0
is_err = meta["is_misregistered"].values.astype(bool)

# item-level scores
p_noisy = meta["p_noisy"].values
score_1mp = 1.0 - p_noisy
# GUARD-informed item score: propagate each item's group C*kappa onto its members,
# combined with the individual mismatch (a simple product; tune in ablation).
grp = pd.read_csv(os.path.join(TAB, "t_layer2_groups.csv")) if os.path.exists(os.path.join(TAB,"t_layer2_groups.csv")) else None

# --- build a GUARD-informed item score if group table is available ---
scores = {"1 - p_noisy": score_1mp}
if grp is not None:
    ck = dict(zip(grp["group"], (grp["C"] * grp["kappa"])))
    group_ck = np.array([ck.get(g_, 0.0) for g_ in meta["noisy_id"].values])
    scores["group C*kappa"] = group_ck
    scores["individual x group"] = score_1mp * (1.0 + group_ck)

fig, ax = plt.subplots(figsize=(5.6, 3.6))
styles = [dict(color=GREYS[0], ls="-"), dict(color=GREYS[2], ls="--"), dict(color=GREYS[1], ls="-.")]
out = []
for (name, s), st in zip(scores.items(), styles):
    b, rec = review_budget_curve(s[valid], is_err[valid])
    ax.plot(b * 100, rec, label=name, **st)
    for budget in (0.05, 0.10, 0.20):
        i = np.argmin(np.abs(b - budget)); out.append((name, budget, rec[i]))
ax.set_xlabel("review budget (%)"); ax.set_ylabel("recall of misregistrations")
ax.legend(frameon=False)
savefig(fig, "f_routing_budget_recall")
pd.DataFrame(out, columns=["score", "budget", "recall"]).to_csv(
    os.path.join(TAB, "t_routing_recall.csv"), index=False)
print("saved f_routing_budget_recall.{png,pdf} and t_routing_recall.csv")

saved f_routing_budget_recall.{png,pdf} and t_routing_recall.csv
